## Faster C-NN Entrenamiento
___

### 1. Ver la clase más grande

Es necesario antes de iniciar el modelo, ver la clase o objeto más grande de las imagenes, esta se encuentra en la carpeta labels y la hacemos para depués poder definir bien la lista de (nose que) que nos permite ejecutar y entrenar el modelo correctamente.

In [3]:
from pathlib import Path

def calcular_num_clases(carpeta_labels):
    carpeta = Path(carpeta_labels)
    clases_encontradas = set()
    
    # Recorremos todos los archivos .txt de la carpeta
    for archivo_txt in carpeta.glob("*.txt"):
        with open(archivo_txt, "r") as f:
            for linea in f:
                partes = linea.strip().split()
                if partes:
                    # La primera columna es el ID de la clase
                    clase_id = int(partes[0])
                    clases_encontradas.add(clase_id)
    
    if not clases_encontradas:
        print("No se encontraron etiquetas en los archivos .txt.")
        return 2  # Mínimo por defecto: 1 clase + 1 fondo
        
    max_clase = max(clases_encontradas)
    # +1 para contar desde el 0 de YOLO + 1 para la casilla del fondo en PyTorch
    num_clases_pytorch = max_clase + 2
    
    print("=== Escaneo de clases ===")
    print(f"Clases únicas encontradas en los .txt: {sorted(list(clases_encontradas))}")
    print(f"ID más alto encontrado: {max_clase}")
    print(f"NUM_CLASES resultante para PyTorch: {num_clases_pytorch}")
    
    return num_clases_pytorch



In [ ]:
import os
import re
from pathlib import Path

import torch
import torchvision
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2,
    FasterRCNN_ResNet50_FPN_V2_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

#=== Variables varias ===
CARPETA_ENTRADA = Path("data/FjordFish/images/train")
CARPETA_LABELS  = Path("data/FjordFish/labels/train")
EXTENSION       = ".jpg"
EPOCHS          = 5
NUM_CLASES = calcular_num_clases(CARPETA_LABELS)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Entrenando en: {device}")


#=== 1. Cargar el modelo e iniciar Fine-Tuning ===

# Cargar el modelo con pesos de COCO
pesos = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn_v2(weights=pesos)

# Reemplazamos la cabeza del clasificador por nuestras clases (Fondo + Pez)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASES)

# FINE-TUNING: Congelamos el backbone (ResNet50) para entrenar solo las capas finales
for param in model.backbone.parameters():
    param.requires_grad = False

model.to(device)


#=== 2. Optimizador (solo entrena los parámetros no congelados) ===
parametros_entrenables = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(parametros_entrenables, lr=0.005, momentum=0.9)


#=== 3. Función simple para leer la etiqueta (.txt) de una imagen ===
def cargar_etiqueta_yolo(ruta_img, carpeta_labels):
    # Leemos la imagen y la escalamos entre 0.0 y 1.0
    img = torchvision.io.read_image(str(ruta_img)).float() / 255.0
    _, alto, ancho = img.shape

    # Buscamos su archivo .txt correspondiente
    ruta_txt = carpeta_labels / f"{ruta_img.stem}.txt"

    cajas = []
    etiquetas = []

    if ruta_txt.exists():
        with open(ruta_txt, "r") as f:
            for linea in f:
                partes = linea.strip().split()
                if len(partes) == 5:
                    clase_yolo, x_c, y_c, w, h = map(float, partes)
                    # Convertimos de YOLO (centro, ancho, alto) a PyTorch (x1, y1, x2, y2)
                    x1 = (x_c - w / 2) * ancho
                    y1 = (y_c - h / 2) * alto
                    x2 = (x_c + w / 2) * ancho
                    y2 = (y_c + h / 2) * alto

                    cajas.append([x1, y1, x2, y2])
                    etiquetas.append(int(clase_yolo) + 1)  # +1 porque el 0 es el fondo

    # Convertimos a Tensores de PyTorch
    if len(cajas) > 0:
        boxes_tensor = torch.tensor(cajas, dtype=torch.float32)
        labels_tensor = torch.tensor(etiquetas, dtype=torch.int64)
    else:
        boxes_tensor = torch.zeros((0, 4), dtype=torch.float32)
        labels_tensor = torch.zeros((0,), dtype=torch.int64)

    target = {"boxes": boxes_tensor, "labels": labels_tensor}
    return img, target


#=== 4. Obtenemos las rutas de las imágenes ===
rutas = sorted([f for f in CARPETA_ENTRADA.iterdir() if f.suffix == EXTENSION])
print(f"Se encontraron {len(rutas)} imágenes para entrenar.")


#=== 5. Bucle de Entrenamiento (5 Épocas) ===

print("\n===== INICIANDO ENTRENAMIENTO =====")

for epoch in range(EPOCHS):
    model.train()  # Activa la capacidad de calcular error y entrenar
    error_acumulado = 0.0

    for ruta in rutas:
        # Cargar la imagen y su etiqueta
        img, target = cargar_etiqueta_yolo(ruta, CARPETA_LABELS)

        # Preparar batch de 1 imagen para el modelo
        batch_imagenes = [img.to(device)]
        batch_targets  = [{k: v.to(device) for k, v in target.items()}]

        # Al estar en model.train(), el modelo nos devuelve el error (loss)
        loss_dict = model(batch_imagenes, batch_targets)
        loss_total = sum(loss for loss in loss_dict.values())

        # Ajustar pesos (Gradiente)
        optimizer.zero_grad()
        loss_total.backward()
        optimizer.step()

        error_acumulado += loss_total.item()

    error_promedio = error_acumulado / len(rutas)
    print(f"Época [{epoch + 1}/{EPOCHS}] - Pérdida promedio (Loss): {error_promedio:.4f}")

#=== 6. Guardamos el resultado del fine-tuning ===
torch.save(model.state_dict(), "fasterrcnn_pesos_finetuned.pth")
print("\n¡Listo! Pesos guardados en 'fasterrcnn_pesos_finetuned.pth'")

=== Escaneo de clases ===
Clases únicas encontradas en los .txt: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 18, 19, 20, 21, 22, 23, 24, 25, 26]
ID más alto encontrado: 26
NUM_CLASES resultante para PyTorch: 28
Entrenando en: cpu
Se encontraron 2086 imágenes para entrenar.

===== INICIANDO ENTRENAMIENTO =====
